# Cleaning data

## Learning objectives

By the end of this notebook you will be able to:

- find and handle missing values: drop, fill or flag
- remove duplicate rows
- standardise text categories
- convert columns to the right types, including dates and numbers stored as text
- detect and handle impossible values and outliers
- write a cleaned dataset

## Introduction

This notebook cleans the laboratory sample register `sample_register_raw.csv`, which has the problems
real registers have: mixed spellings, blanks, results stored as text, an impossible temperature code and
duplicated rows. Every step is code, so the cleaning is documented and repeatable.

In [1]:
import pandas as pd
import numpy as np
RAW = "../datasets/raw/"
PROCESSED = "../datasets/processed/"

In [2]:
raw = pd.read_csv(RAW + "sample_register_raw.csv")
print(raw.shape)
raw.head(8)

(430, 8)


,sample_id,patient_id,collected,sample_type,result,concentration,temperature_c,technician
0,S-0001,P0285,14/07/2025,blood,Negative,>100,4.2,BK
1,S-0002,P0225,01/05/2025,urine,Negative,3.2,5.7,CM
2,S-0003,P0067,05/03/2025,urine,Negative,22.0,2.4,BK
3,S-0004,P0218,10/01/2025,BLOOD,NEG,8.0,3.9,BK
4,S-0005,P0195,14/12/2025,BLOOD,NEG,8.0,4.8,BK
5,S-0006,P0254,08/06/2025,urine,Negative,not done,99.0,an
6,S-0007,P0149,10/06/2025,BLOOD,neg,3.2,4.4,CM
7,S-0008,P0090,09/06/2025,blood,Positive,22.0,2.7,CM


Work on a copy so the raw data stays available for comparison.

In [3]:
df = raw.copy()
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 430 entries, 0 to 429
Data columns (total 8 columns):
 #   Column         Non-Null Count  Dtype  
---  ------         --------------  -----  
 0   sample_id      430 non-null    str    
 1   patient_id     430 non-null    str    
 2   collected      430 non-null    str    
 3   sample_type    430 non-null    str    
 4   result         410 non-null    str    
 5   concentration  430 non-null    str    
 6   temperature_c  430 non-null    float64
 7   technician     396 non-null    str    
dtypes: float64(1), str(7)
memory usage: 43.5 KB


## 1. Duplicates

In [4]:
print(df.duplicated().sum(), "duplicated rows")
df = df.drop_duplicates()
print(df.shape, "| sample IDs unique now:", df["sample_id"].is_unique)

10 duplicated rows
(420, 8) | sample IDs unique now: True


`drop_duplicates()` keeps the first copy of rows that are identical in every column. With
`subset=["sample_id"]` it would compare only that column. Check that the remaining IDs are unique, as
here.

## 2. Standardising text

In [5]:
print(df["sample_type"].value_counts())

sample_type
urine     83
Saliva    76
Blood     74
blood     71
BLOOD     64
Urine     52
Name: count, dtype: int64


In [6]:
df["sample_type"] = df["sample_type"].str.strip().str.capitalize()
df["sample_type"].value_counts()

sample_type
Blood     209
Urine     135
Saliva     76
Name: count, dtype: int64

In [7]:
print(df["result"].value_counts(dropna=False))

result
Negative    203
Positive     92
NEG          40
neg          39
positive     26
NaN          20
Name: count, dtype: int64


For categories with different words for the same thing, a mapping dictionary is clearest. Anything not
in the map becomes NaN, so unexpected spellings show up instead of slipping through.

In [8]:
result_map = {"positive": "Positive", "negative": "Negative", "neg": "Negative"}
df["result"] = df["result"].str.strip().str.lower().map(result_map)
df["result"].value_counts(dropna=False)

result
Negative    282
Positive    118
NaN          20
Name: count, dtype: int64

In [9]:
df["technician"] = df["technician"].str.upper()
df["technician"].value_counts(dropna=False)

technician
AN     162
BK     115
CM     110
NaN     33
Name: count, dtype: int64

## 3. Dates

In [10]:
print(df["collected"].head(3).tolist())
df["collected"] = pd.to_datetime(df["collected"], format="%d/%m/%Y")
print(df["collected"].dtype, df["collected"].min().date(), df["collected"].max().date())

['14/07/2025', '01/05/2025', '05/03/2025']
datetime64[us] 2025-01-01 2025-12-30


Giving `format=` avoids ambiguity: is `01/05/2025` the 1st of May or the 5th of January? Here the file
uses day/month/year. If some entries do not match, `errors="coerce"` turns them into missing values
(`NaT`) instead of stopping.

## 4. Numbers stored as text

The concentration column mixes numbers with `<0.5` (below the detection limit), `>100` (above the
upper limit) and `not done`.

In [11]:
print(df["concentration"].value_counts().head(10))

concentration
<0.5        73
3.2         55
8.0         55
>100        51
22.0        49
12.5        49
15.1        45
not done    43
Name: count, dtype: int64


A common approach: keep a flag saying whether a value was censored, and store a numeric value for
analysis. Here values below the limit are set to half the limit, a simple convention; values above the
limit are set to the limit itself. Whatever rule you choose, record it.

In [12]:
conc = df["concentration"].str.strip()
df["conc_flag"] = np.select([conc.str.startswith("<"), conc.str.startswith(">"), conc == "not done"],
                            ["below_limit", "above_limit", "not_done"], default="measured")
numeric = pd.to_numeric(conc.str.lstrip("<>"), errors="coerce")
df["concentration_num"] = np.where(df["conc_flag"] == "below_limit", numeric / 2, numeric)
df[["concentration", "conc_flag", "concentration_num"]].drop_duplicates().head(8)

,concentration,conc_flag,concentration_num
0,>100,above_limit,100.00
1,3.2,measured,3.20
2,22.0,measured,22.00
3,8.0,measured,8.00
5,not done,not_done,NaN
9,15.1,measured,15.10
11,<0.5,below_limit,0.25
23,12.5,measured,12.50


In [13]:
df["conc_flag"].value_counts()

conc_flag
measured       253
below_limit     73
above_limit     51
not_done        43
Name: count, dtype: int64

`pd.to_numeric(..., errors="coerce")` converts what it can and turns the rest (`not done`) into NaN.

## 5. Impossible values

In [14]:
df["temperature_c"].describe().round(2)

count    420.00
mean       5.33
std       11.35
min        0.50
25%        3.10
50%        4.10
75%        4.80
max       99.00
Name: temperature_c, dtype: float64

In [15]:
print((df["temperature_c"] == 99.0).sum(), "readings of exactly 99.0")
print(df["temperature_c"].sort_values().tail(8).tolist())

6 readings of exactly 99.0
[6.8, 6.8, 99.0, 99.0, 99.0, 99.0, 99.0, 99.0]


Every other reading is below 7 °C, then there is a jump to exactly 99.0 °C, several times over. That is a
code for "not recorded", not a real storage temperature. The mean and standard deviation above are
badly distorted by it. Replace it with NaN:

In [16]:
df["temperature_c"] = df["temperature_c"].replace(99.0, np.nan)
df["temperature_c"].describe().round(2)

count    414.00
mean       3.97
std        1.19
min        0.50
25%        3.10
50%        4.10
75%        4.80
max        6.80
Name: temperature_c, dtype: float64

Now flag samples stored outside the acceptable 2-8 °C range:

In [17]:
df["temp_ok"] = df["temperature_c"].between(2, 8)
print(df["temp_ok"].value_counts())
df.loc[~df["temp_ok"] & df["temperature_c"].notna(), ["sample_id", "temperature_c"]].head()

temp_ok
True     391
False     29
Name: count, dtype: int64


,sample_id,temperature_c
16,S-0017,1.1
26,S-0027,1.8
37,S-0038,1.9
62,S-0063,1.7
93,S-0094,1.6


## 6. Outliers

An *outlier* is a value far from the rest. It may be an error or a real extreme; investigate before
removing. A standard rule flags values more than 1.5 × IQR below the first quartile or above the
third quartile (the box-plot rule).

In [18]:
def iqr_flags(s, k=1.5):
    q1, q3 = s.quantile([0.25, 0.75])
    iqr = q3 - q1
    return (s < q1 - k * iqr) | (s > q3 + k * iqr)

measured = df.loc[df["conc_flag"] == "measured", "concentration_num"]
print(measured.describe().round(2))
print(iqr_flags(measured).sum(), "measured concentrations flagged by the IQR rule")

count    253.00
mean      11.80
std        6.45
min        3.20
25%        8.00
50%       12.50
75%       15.10
max       22.00
Name: concentration_num, dtype: float64
0 measured concentrations flagged by the IQR rule


None are flagged: the measured values all lie between the detection limits, in a fairly narrow range.
On other columns or datasets, look at the flagged rows before deciding whether to correct, keep or
exclude them, and record the decision.

## 7. Missing values

Options, and when each fits:

| approach | code | use when |
|---|---|---|
| keep and report | `isna().sum()` | always: know how much is missing |
| drop rows | `dropna(subset=[...])` | few missing, and missingness unrelated to the outcome |
| fill with a value | `fillna(value)` | a meaningful default exists ("Unknown") |
| fill with a statistic | `fillna(s.median())` | quick descriptive work; it understates variability |
| flag | `s.isna()` as a new column | to analyse who is missing |

In [19]:
df.isna().sum()

sample_id             0
patient_id            0
collected             0
sample_type           0
result               20
concentration         0
temperature_c         6
technician           33
conc_flag             0
concentration_num    43
temp_ok               0
dtype: int64

In [20]:
df["technician"] = df["technician"].fillna("UNKNOWN")
print(df["result"].isna().sum(), "results missing: kept as missing, not guessed")
complete_results = df.dropna(subset=["result"])
print(len(df), "rows;", len(complete_results), "with a result")

20 results missing: kept as missing, not guessed
420 rows; 400 with a result


Missing test results are kept as missing: inventing a result would be wrong. Analyses of positivity use
`complete_results` and report how many were excluded.

## 8. Types and tidy columns

In [21]:
df["sample_type"] = df["sample_type"].astype("category")
df["result"] = df["result"].astype("category")
df = df.drop(columns=["concentration"]).rename(columns={"concentration_num": "concentration"})
df.dtypes

sample_id                   str
patient_id                  str
collected        datetime64[us]
sample_type            category
result                 category
temperature_c           float64
technician                  str
conc_flag                   str
concentration           float64
temp_ok                    bool
dtype: object

## 9. Check and save

In [22]:
assert df["sample_id"].is_unique
assert set(df["sample_type"].dropna()) <= {"Blood", "Urine", "Saliva"}
assert df["temperature_c"].dropna().lt(50).all()
print(f"raw {raw.shape} -> clean {df.shape}")

df.to_csv(PROCESSED + "sample_register_clean.csv", index=False)
pd.read_csv(PROCESSED + "sample_register_clean.csv").head()

raw (430, 8) -> clean (420, 10)


,sample_id,patient_id,collected,sample_type,result,temperature_c,technician,conc_flag,concentration,temp_ok
0,S-0001,P0285,2025-07-14,Blood,Negative,4.2,BK,above_limit,100.0,True
1,S-0002,P0225,2025-05-01,Urine,Negative,5.7,CM,measured,3.2,True
2,S-0003,P0067,2025-03-05,Urine,Negative,2.4,BK,measured,22.0,True
3,S-0004,P0218,2025-01-10,Blood,Negative,3.9,BK,measured,8.0,True
4,S-0005,P0195,2025-12-14,Blood,Negative,4.8,BK,measured,8.0,True


The `assert` lines make the cleaning self-checking: if a future version of the raw file has a new
spelling or a new error code, the notebook stops and tells you.

## A summary of the cleaning log

| problem | action |
|---|---|
| duplicated rows | removed |
| sample type in mixed case with spaces | stripped and capitalised |
| result in five spellings | mapped to Positive / Negative; blanks kept missing |
| technician initials in mixed case | upper case; blanks set to UNKNOWN |
| dates as day/month/year text | converted to dates |
| concentration as text with `<`, `>`, `not done` | flag column plus numeric value |
| temperature code 99.0 | set to missing; range flag added |

## Practise

Exercises for this folder are in `exercises/Pandas_Exercises.ipynb`.

## Further reading

- [pandas user guide](https://pandas.pydata.org/docs/user_guide/index.html)
- [Working with missing data](https://pandas.pydata.org/docs/user_guide/missing_data.html)
- [Working with text data](https://pandas.pydata.org/docs/user_guide/text.html)
- [Hadley Wickham, Tidy data (2014)](https://doi.org/10.18637/jss.v059.i10)